# **BERT Text Classification**

In this session, we will explore how to use BERT (Bidirectional Encoder Representations from Transformers) for text classification tasks. BERT is a powerful pre-trained language model that has achieved state-of-the-art results in various natural language processing tasks, including text classification.

We will go through the training process of a BERT model for text classification, including data preparation, model architecture, and evaluation.

## Data Preparation

Before we can train a BERT model for text classification, we need to prepare our dataset. This involves several steps, including loading the data, preprocessing the text, and creating input features that can be fed into the BERT model.

We will use the imdb movie reviews dataset for this example, which is a commonly used dataset for text classification tasks. The dataset consists of movie reviews labeled as positive or negative.

We will use 1.000 samples from the dataset train set for this example. 

In [1]:
from datasets import load_dataset
import pandas as pd

imdb = load_dataset("imdb", split="train[:1000]")

df = pd.DataFrame({"content": imdb["text"], "label": imdb["label"]})
df

,content,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
995,WARNING SPOILERS***** A really stupid movie ab...,0
996,"For some reason, various young couples hiking ...",0
997,This film is exactly what you get when you rea...,0
998,I rented this one to see Vanesa Talor one more...,0


Next we will clean the data using spaCy library. We will remove stop words, punctuation, and lemmatize the text to prepare the text for BERT input.

In [2]:
MODEL_NAME = "en_core_web_sm"

!pip install spacy
!python -m spacy download $MODEL_NAME

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 8.9 MB/s  0:00:019.0 MB/s eta 0:00:0101
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
import spacy

nlp = spacy.load(MODEL_NAME)


def clean(text: str) -> str:
    doc = nlp(text)
    tokens = [
        token.lemma_  # Lemmatization
        for token in doc
        if not token.is_stop  # Remove stop words
        and not token.is_punct  # Remove punctuation
    ]

    return " ".join(tokens)

In [4]:
df["content_cleaned"] = df["content"].apply(clean)
df.head()

,content,label,content_cleaned
0,I rented I AM CURIOUS-YELLOW from my video sto...,0,rent CURIOUS yellow video store controversy su...
1,"""I Am Curious: Yellow"" is a risible and preten...",0,curious Yellow risible pretentious steaming pi...
2,If only to avoid making this type of film in t...,0,avoid make type film future film interesting e...
3,This film was probably inspired by Godard's Ma...,0,film probably inspire Godard Masculin féminin ...
4,"Oh, brother...after hearing about this ridicul...",0,oh brother hear ridiculous film umpteen year t...


In [5]:
review_path = "../dataset/review_cleaned.csv"
df[["content", "content_cleaned", "label"]].to_csv(review_path, index=False)

## Data Splitting

Before we can train our model, we need to split our dataset into training and validation sets. This allows us to evaluate the performance of our model on unseen data during the training process. We will use an 80-20 split for our training and validation sets.

We can use the `train_test_split` function from the `sklearn` library to perform this split. This function will randomly shuffle the data and split it into the specified proportions for training and validation.

In [6]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and test sets
contents = df[["content", "content_cleaned"]]
labels = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    contents, labels, test_size=0.2, random_state=42
)

# Save the test sets to CSV files
test_path = "../dataset/review_final_test.csv"
test_df = pd.DataFrame(
    {
        "content": X_test["content"],
        "content_cleaned": X_test["content_cleaned"],
        "label": y_test,
    }
)
test_df.to_csv(test_path, index=False)

# Split the training set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Save the training and validation sets to CSV files
train_path = "../dataset/review_final_train.csv"
train_df = pd.DataFrame(
    {
        "content": X_train["content"],
        "content_cleaned": X_train["content_cleaned"],
        "label": y_train,
    }
)
train_df.to_csv(train_path, index=False)

val_path = "../dataset/review_final_val.csv"
val_df = pd.DataFrame(
    {
        "content": X_val["content"],
        "content_cleaned": X_val["content_cleaned"],
        "label": y_val,
    }
)
val_df.to_csv(val_path, index=False)

With this preparation done, we are now ready to move on to the next step, which is building and training our BERT model for text classification.

## Model Training

Here will train `answerdotai/ModernBERT-base` model for text classification using the Hugging Face Transformers library. We will fine-tune the pre-trained BERT model on our dataset to adapt it to our specific task of classifying movie reviews as positive or negative.

In [7]:
!pip install transformers accelerate datasets

In [8]:
from transformers import (
    logging,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from datasets import Dataset

logging.set_verbosity_error()

In [18]:
MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)


# Tokenizer function
def tokenize(texts):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors=None,
    )


# Prepare the datasets for training
train_df = pd.read_csv(train_path)
train_tokenized = tokenize(train_df["content_cleaned"].tolist())
train_tokenized["label"] = train_df["label"].tolist()

eval_df = pd.read_csv(val_path)
eval_tokenized = tokenize(eval_df["content_cleaned"].tolist())
eval_tokenized["label"] = eval_df["label"].tolist()

train_dataset = Dataset.from_dict(train_tokenized)
eval_dataset = Dataset.from_dict(eval_tokenized)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

After we prepare the data, we will create a `Trainer` object from the Hugging Face library, which will handle the training loop for us. We will specify the training arguments, such as the number of epochs, batch size, and learning rate.

Finally, we will call the `train` method on the `Trainer` object to start the training process. After training is complete, we can evaluate our model on the validation set to see how well it performs on unseen data.

In [19]:
# Create the Trainer
training_args = TrainingArguments(
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    eval_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [20]:
# Save the trained model
model_path = "../model/bert_text_classification"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../model/bert_text_classification/tokenizer_config.json',
 '../model/bert_text_classification/tokenizer.json')

# Model Inference

We then can use the trained model to make predictions on new, unseen data. This involves preprocessing the input text in the same way we did for the training data, and then feeding it into the model to get the predicted class label (positive or negative).

In [12]:
!pip install protobuf sentencepiece tiktoken

In [21]:
from transformers import pipeline

# Load the trained model for inference
model_path = "../model/bert_text_classification"
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)

classifier = pipeline(
    "text-classification",
    model=loaded_model,
    tokenizer=loaded_tokenizer,
)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

In [ ]:
# Load the test dataset
test_df = pd.read_csv(test_path)

# Perform inference on the test dataset
predictions = classifier(
    test_df["content_cleaned"].tolist(), truncation=True, max_length=512
)

As we still has different label between our initial label and the label from the model, we can map the predicted labels to our original labels for better interpretation of the results.

In [ ]:
test_df["predicted_label"] = [pred["label"] for pred in predictions]
test_df["label"] = test_df["label"].apply(
    lambda x: "positive" if x == 1 else "negative"
)

results_path = "../dataset/review_inference_result.csv"
test_df.to_csv(results_path, index=False)

test_df.head()

,content,content_cleaned,label,predicted_label
0,This movie could be used in film classes in a ...,movie film class script B Movie course inheren...,negative,positive
1,Prom Night is about a girl named Donna (Britta...,Prom Night girl name Donna Brittany Snow chase...,negative,positive
2,"When I first saw the trailer for Prom Night, I...",see trailer Prom Night admit trailer look good...,negative,positive
3,Finally i thought someone is going to do justi...,finally think go justice H.G. Wells classic ve...,negative,positive
4,I was disgusted by this movie. No it wasn't be...,disgust movie graphic sex scene ruin image Art...,negative,positive


Then we can evaluate the performance of our model on the test dataset by comparing the predicted labels with the true labels. We can calculate metrics such as accuracy, precision, recall, and F1-score to assess how well our model is performing.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


def evaluate_perlabel(
    y_true: list[str],
    y_pred: list[str],
    label: str,
) -> dict[str, float]:

    y_true_binary = [1 if y == label else 0 for y in y_true]
    y_pred_binary = [1 if y == label else 0 for y in y_pred]

    accuracy = accuracy_score(y_true=y_true_binary, y_pred=y_pred_binary)
    precision = precision_score(
        y_true=y_true_binary, y_pred=y_pred_binary, zero_division=0
    )
    recall = recall_score(y_true=y_true_binary, y_pred=y_pred_binary, zero_division=0)
    f1 = f1_score(
        y_true=y_true_binary, y_pred=y_pred_binary, pos_label=1, zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
    }

In [28]:
y_true = test_df["label"].tolist()
y_pred = test_df["predicted_label"].tolist()

print("Evaluation for positive class:")
positive_metrics = evaluate_perlabel(y_true, y_pred, label="positive")
for metric, value in positive_metrics.items():
    print(f"{metric.capitalize()}: {value:.4f}")

print("\nEvaluation for negative class:")
negative_metrics = evaluate_perlabel(y_true, y_pred, label="negative")
for metric, value in negative_metrics.items():
    print(f"{metric.capitalize()}: {value:.4f}")

Evaluation for positive class:
Accuracy: 0.0000
Precision: 0.0000
Recall: 0.0000
F1_score: 0.0000

Evaluation for negative class:
Accuracy: 0.0000
Precision: 0.0000
Recall: 0.0000
F1_score: 0.0000


We can see the results is exceptionally bad, where all the metrics are 0. This is because the model is not able to correctly classify any of the samples in the test set, which indicates that the model has not learned to generalize well from the training data. This could be due to various reasons such as insufficient training data, overfitting, or a mismatch between the training and test data distributions.